# 08 Robustness of the measure and ablations
Repeats the reference analysis under other thresholds, null tolerances, eccentricity ranges and binnings, without the cortex near the meridians, with the default sampling threshold and a finer step of the tracker, on each pRF half and on cortical subdivisions, and summarises the ablation networks. Only the tracker settings track again; everything else re-scores the cached tractograms.

In [ ]:
DEV_MODE = True    # True: small test run (dev: block of config.yaml). False: full experiment (full: block).

import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from retinopath.core import load_config, get_device, seed_all, Timer
from retinopath import pipeline as PL

cfg = load_config(DEV_MODE); device = get_device(); seed_all(cfg.seed)
RES = cfg.P.out / "results"; FIG = cfg.P.out / "figures"; RES.mkdir(exist_ok=True); FIG.mkdir(exist_ok=True)
print(f"DEV_MODE={DEV_MODE} | device={device} | outputs -> {cfg.P.out}")
if DEV_MODE: print("DEV_MODE: small test run. Its outputs are kept apart in outputs/dev and are not experiment results.")

In [ ]:
import copy
from retinopath import metrics as M
subjects = PL.make_subject_lists(cfg); test = [PL.session(sid) for sid in subjects["test"]]
# The finer step keeps the tightest allowed curve of the main tracker: step / (2 sin(max_angle / 2)) is the same radius.
fine_step_mm = 0.2
fine_angle = round(float(np.degrees(2 * np.arcsin(fine_step_mm / cfg.tracking.step_mm * np.sin(np.radians(cfg.tracking.max_angle) / 2)))), 1)
variants = {"default": {}, "r2>=5": {"prf.r2_min": 5}, "r2>=20": {"prf.r2_min": 20}, "null tol 5%": {"prf.null_tol": 0.05}, "null tol 20%": {"prf.null_tol": 0.20},
            "ecc<=5": {"prf.ecc_max": 5.0}, "ecc>=1": {"prf.ecc_min": 1.0},
            "ecc 1-6, 10 bins": {"prf.ecc_min": 1.0, "prf.ecc_max": 6.0, "prf.n_ecc_bins": 10},
            "meridian margin 15 deg": {"prf.meridian_margin_deg": 15},
            "sampling threshold 0.1": {"tracking.pmf_threshold": 0.1},
            f"step {fine_step_mm} mm": {"tracking.step_mm": fine_step_mm, "tracking.max_angle": fine_angle}}
with Timer(cfg, "08_robustness"):
    rows = [row for session_rows in PL.each(cfg, lambda worker_cfg, session: PL.robustness_rows(worker_cfg, session, variants), test, "robustness") for row in session_rows]
robustness = pd.DataFrame(rows); robustness.to_csv(RES / "robustness.csv", index=False)
robustness.groupby("analysis", sort=False)[["n_valid", "rsi_ree_all", "rsi_ree_V1-V2", "rsi_ree_V2-V3", "rsi_ree_V1-V3", "auc_binned_V1-V2", "auc_binned_V2-V3", "auc_binned_V1-V3"]].median().round(3)

In [ ]:
# Ablations: networks trained in notebook 05, scored in notebook 06
learned_file = RES / "learned.csv"
if learned_file.exists():
    learned = pd.read_csv(learned_file); learned["variant"] = learned.method.str.extract(r"net_(.*)_s\d+")[0]
    display(learned.groupby(["variant", "protocol"])[["rsi_ree_all", "acc", "peak_err_deg", "n_pairs"]].median().round(3).unstack())
else:
    print("run notebook 06 first")